**IMPROVEMENTS**

* [] pass the json for validation directly to the model invocation when possible (check if it's supported) i.e. stop to do the validation after the invocation

Self healing strategies:

* [x] increase temperature error after error (already implemented, works sometimes for qwen3)
* [ ] pass the error message when problems in the json are detected to see if it fixes them afterwards (new prompt with schema, invalid JSON, validation error and "Fix the JSON so it passes validation. Do not change values unless required to fix an error.")

In [1]:
import sys
from groq import Groq
import json
import pydantic 
from pydantic import BaseModel
from typing import get_args, Literal
import os
from google import genai
from openai import OpenAI
from time import sleep

from prompts import ORIGINAL_SYSTEM_PROMPT
from dataset_utils import load_or_download_dataset
from models import build_crf_model
from crf_utils import build_compact_schema, save_prediction, create_base_messages, invoke_model
from api_keys import *

In [2]:
LANGUAGE = "it"
# MODEL_ID = "qwen/qwen3-32b"
# MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
# MODEL_ID = "openai/gpt-oss-120b"
# MODEL_ID = "gemma-3-27b-it"
MODEL_ID = "gemini-3-flash-preview" # "gemini-3-pro-review"
# MODEL_ID = "gpt-5-nano"
if "gemini" in MODEL_ID:
    LLM_TYPE = "gemini"
elif "gemma" in MODEL_ID:
    LLM_TYPE = "gemma"
elif "gpt" in MODEL_ID and "oss" not in MODEL_ID:
    LLM_TYPE = "openai"
else:
    LLM_TYPE = "groq"
OUTPUT_FILE = f"{MODEL_ID.split('/')[-1]}_{LANGUAGE}.jsonl"
OUTPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", OUTPUT_FILE)

FS_DATASET = "NLP-FBK/dyspnea-crf-train"
DATASET_TO_CRF = "NLP-FBK/dyspnea-crf-test"
NUM_FS_EXAMPLES = 3
MAX_RETRIES = 3

MODEL_ID, LLM_TYPE

('gemini-3-flash-preview', 'gemini')

In [3]:
os.chdir("..")

In [ ]:
# one-time initializations
if LLM_TYPE=="groq":
    client = Groq(api_key=GROQ_API_KEY)
elif LLM_TYPE in ["gemma", "gemini"]:
    GOOGLE_API_KEYS = [
    ]
    api_key_index = 0
    client = genai.Client(api_key=GOOGLE_API_KEYS[api_key_index])
elif LLM_TYPE=="openai":
    client = OpenAI(api_key=OPENAI_API_KEY)
CRFModel = build_crf_model()

# Load development dataset
ds = load_or_download_dataset(DATASET_TO_CRF)[LANGUAGE]
print(f"{DATASET_TO_CRF} has {len(ds)} documents")

base_messages, sys_instruction = create_base_messages(CRFModel, ORIGINAL_SYSTEM_PROMPT, FS_DATASET, LANGUAGE, NUM_FS_EXAMPLES, llm_type=LLM_TYPE)
base_messages[-1]

Found local cache at data/dyspnea-valid-options, loading from disk...
Dataset({
    features: ['options', 'item'],
    num_rows: 134
})
Found local cache at data/dyspnea-crf-test, loading from disk...
NLP-FBK/dyspnea-crf-test has 200 documents
Found local cache at data/dyspnea-crf-train, loading from disk...


Content(
  parts=[
    Part(
      text='{"history of recent trauma": "y", "cardiovascular diseases": "y"}'
    ),
  ],
  role='model'
)

In [5]:
# in case multiple runs are required for the same elaboraiton
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["document_id"] for line in f}
len(processed_ids), processed_ids

(200,
 {'1000632_it',
  '1003423_it',
  '1009021_it',
  '10095_it',
  '1015098_it',
  '1015981_it',
  '1017129_it',
  '1025349_it',
  '1037766_it',
  '1041183_it',
  '1051592_it',
  '105476_it',
  '1071944_it',
  '1078357_it',
  '1084060_it',
  '1085357_it',
  '1101309_it',
  '1103640_it',
  '111414_it',
  '1115875_it',
  '1124732_it',
  '1143513_it',
  '1160226_it',
  '1181202_it',
  '1187128_it',
  '1189593_it',
  '1190319_it',
  '1201050_it',
  '1203238_it',
  '1209704_it',
  '121138_it',
  '1224596_it',
  '1229118_it',
  '1233677_it',
  '1234598_it',
  '1242704_it',
  '1255328_it',
  '1257087_it',
  '1259590_it',
  '1266671_it',
  '127947_it',
  '1281662_it',
  '1286408_it',
  '1293485_it',
  '12953_it',
  '1318160_it',
  '132042_it',
  '1328625_it',
  '134411_it',
  '1345233_it',
  '1360231_it',
  '1379086_it',
  '1387943_it',
  '1391017_it',
  '1411975_it',
  '1413438_it',
  '141756_it',
  '1420059_it',
  '1423237_it',
  '1434391_it',
  '1440707_it',
  '1448461_it',
  '1456139_it

In [ ]:
#TODO clean this function
def crf_dataset():
    global client, api_key_index
    with open(OUTPUT_PATH, "a") as f: #TODO this is append now
        for item in ds:
            doc_id = item["document_id"]
            clinical_note = item["clinical_note"]
            if doc_id in processed_ids:
                print(f"Skipped {doc_id} because already processed")
                continue
            success = False
            
            print(f"Processing document: {doc_id}")

            for attempt in range(MAX_RETRIES):
                if "gemma" in MODEL_ID:
                    sleep(20)

                try:
                    # 1. Invoke model
                    output = invoke_model(
                        client,
                        MODEL_ID,
                        final_message=f"Now process the following clinical note:\n{clinical_note}",
                        messages=base_messages.copy(),
                        attempt=attempt,
                        system_instruction=sys_instruction,
                        pydantic_model=CRFModel
                    )

                    if isinstance(output, BaseModel):
                        result = output
                    else:
                        # 2. Parse JSON string
                        data_dict = json.loads(output)
                        # 3. Validate with Pydantic
                        result = CRFModel.model_validate(data_dict)
                    
                    # 4. Save
                    save_prediction(f, doc_id, result)
                    success = True
                    break # Exit retry loop on success
                except json.JSONDecodeError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] JSON Error for {doc_id}: {e}")
                    print(f"FAILED OUTPUT:\n{output}\n")
                    # Retry
                
                except pydantic.ValidationError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Validation Error for {doc_id}:")
                    print(e)
                    print(f"FAILED OUTPUT:\n{output}\n")
                    # Retry
                
                except Exception as e:
                    if "The model is overloaded" in str(e) or "This model is currently experiencing high demand" in str(e):
                        print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Service Unavailable for {doc_id}: {e}")
                        attempt-=1
                        sleep(15)
                        continue
                    if "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits" in str(e):
                        print(f"Daily calls are over (with {doc_id}): {e}")
                        attempt-=1
                        api_key_index+=1
                        if api_key_index > len(GOOGLE_API_KEYS)-1:
                            return
                        print("New Google API key used: ", GOOGLE_API_KEYS[api_key_index])
                        client = genai.Client(api_key=GOOGLE_API_KEYS[api_key_index])
                        continue
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] System/API Error for {doc_id}: {e}")
                    return # TODO currently stopping in case the tokens for that day are over

            if not success:
                print(f"Skipping document {doc_id} after {MAX_RETRIES} failed attempts.")
                
                # Fallback: Create fake instance with all "unknown"
                fake_instance = CRFModel()
                save_prediction(f, doc_id, fake_instance)
        print("Done")


crf_dataset()

Skipped 1003423_it because already processed
Skipped 1051592_it because already processed
Skipped 1160226_it because already processed
Skipped 121138_it because already processed
Skipped 1224596_it because already processed
Skipped 1229118_it because already processed
Skipped 1286408_it because already processed
Skipped 1448461_it because already processed
Skipped 1466535_it because already processed
Skipped 1487935_it because already processed
Skipped 1509420_it because already processed
Skipped 1594629_it because already processed
Skipped 1744582_it because already processed
Skipped 1750279_it because already processed
Skipped 1754455_it because already processed
Skipped 199524_it because already processed
Skipped 249048_it because already processed
Skipped 340214_it because already processed
Skipped 352393_it because already processed
Skipped 354519_it because already processed
Skipped 459381_it because already processed
Skipped 508770_it because already processed
Skipped 568647_it 

In [ ]:
os.chdir("CRF-filling-CL4Health2026")
!python3 scoring.py --submission_path your_submission_data/{OUTPUT_FILE} --language {LANGUAGE}


=== Scoring program starting ===
Loading reference from development_data/dev_gt.jsonl
Loading submission from your_submission_data/gemini-3-flash-preview_it.jsonl
Traceback (most recent call last):
  File "/Users/niccolomorabito/Desktop/CRF task/CRF-filling-CL4Health2026/scoring.py", line 136, in <module>
    main(your_submission_path, language, test_or_dev="development")
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/niccolomorabito/Desktop/CRF task/CRF-filling-CL4Health2026/scoring.py", line 111, in main
    score = scorer.calculate_score(reference, submission)
  File "/Users/niccolomorabito/Desktop/CRF task/CRF-filling-CL4Health2026/scoring.py", line 36, in calculate_score
    raise ValueError(
        f"Document ID mismatch: reference {ref_one_patient['document_id']} vs submission {sub_one_patient['document_id']}"
    )
ValueError: Document ID mismatch: reference 1014081 vs submission 1003423_it
